# Executive Summary: QUBO Input Package Generator (V6 + Anchor-City Customization)

### Project Context
Refactor of `data_generation_V5_3.ipynb` for the **Supply Chain Network
Optimization and Quantum Technology Evaluation** capstone with Dell
Technologies and ASU. Produces synthetic Forward Stocking Location (FSL)
network instances for PyQUBO-based solvers.

### What changed vs. V5_3
1. **No more hidden literals.** Every dial — network scale, spatial band
   probs, distributions, all `lambda_*` weights, SLA thresholds — is driven
   by `batch_runs.csv`.
2. **Base package contains facts only.** `d_ij` is raw haversine distance,
   `20` / `130` / `180` are promoted to `base_miles`, `penalty_start_miles`,
   `max_service_miles` in `parameters.csv`.
3. **Anchor cities live in external CSV files** (new in this revision).
   `configs/anchor_cities/default_anchor_cities.csv` is the default universe
   of eligible hub locations. Each `batch_runs.csv` row can override it via
   the optional `anchor_city_file` column.

### Anchor-city design (must-read)
- Anchor cities define the **eligible universe** of hub locations.
- Each generated hub corresponds to **exactly one anchor city**.
- Sampling is **without replacement** — no more than one hub per city per
  instance.
- Larger `n_hubs` is supported by **adding more rows to the anchor-city
  CSV**, not by duplicating cities or jittering coordinates.
- Hub coordinates and `region_code` are read directly from the anchor CSV;
  there is no geographic inference inside the hub generator.

### Output per instance (unchanged base package)
```
outputs/<instance_name>/
    parameters.csv              # one row, global scalars only
    hubs.csv                    # hub_id, anchor_id, anchor_city, lat, lon,
                                # region_code, T_j, B_j
    parts.csv                   # part_id, P_k
    zips.csv                    # zip_id, lat, lon, region_code
    demand.csv                  # zip_id, part_id, Q_ik, b_ik
    distances.csv               # zip_id, hub_id, d_ij  (raw miles)
    optional_baseline_part_homes.csv
    optional_parameter_key.csv
    summary_report.csv          # now also records anchor_city_file + counts
```


## Section 1: Environment Setup and Library Imports

Same stack as V5_3 plus `dataclasses` and `argparse`.

In [1]:
from __future__ import annotations

import argparse
import math
import shutil
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd

## Section 2: Anchor-City Configuration

Anchor cities used to live as a 100-entry `ANCHOR_CITIES` list literal in the
notebook. They have been **moved out of code into CSV files** under
`configs/anchor_cities/`. This section defines only the path / region
constants — the real list lives in
`configs/anchor_cities/default_anchor_cities.csv` and can be swapped per
instance via the optional `anchor_city_file` batch column.

### Why
- **Scenario customization** — e.g. a LATAM-only scenario is a separate
  CSV, not a code branch.
- **Scales cleanly** — supporting more hubs is a matter of adding rows to
  the CSV, not editing Python.
- **Auditable** — every instance records which anchor file it consumed in
  `summary_report.csv`.

### Rules
- Required anchor columns: `anchor_id`, `city`, `lat`, `lon`, `region_code`.
- `anchor_id` is the unique key (city names alone are ambiguous — Portland,
  OR vs. Portland, ME).
- Allowed `region_code` values: `NAM`, `LATAM`, `EMEA`, `APAC`.
- Hubs are sampled **without replacement** (one hub per city per instance).


In [15]:
DEFAULT_ANCHOR_CITY_FILE = "/content/default_anchor_cities.csv"

ALLOWED_REGION_CODES = {"NAM", "LATAM", "EMEA", "APAC"}

REQUIRED_ANCHOR_COLUMNS = ("anchor_id", "city", "lat", "lon", "region_code")

# Retained so infer_region() can still tag LATAM cities correctly when used
# as a standalone utility. The generator itself no longer relies on this --
# region_code now comes directly from the anchor-city CSV.
LATAM_ANCHOR_CITIES = {
    "Mexico City", "Sao Paulo", "Buenos Aires", "Bogota", "Lima",
    "Santiago", "Caracas", "Rio de Janeiro", "Guadalajara", "Monterrey",
}

## Section 3: `GeneratorConfig` and `ModelParams` (Parameter Model)

Two dataclasses — same split as before. The only new field is
`anchor_city_file` on `GeneratorConfig`: an optional per-instance override of
the default anchor CSV path.


In [3]:
@dataclass
class GeneratorConfig:
    instance_name: str
    seed: int = 42
    schema_version: str = "1.0.0"

    n_hubs: int = 40
    n_zips: int = 1000
    n_parts: int = 150

    candidate_fraction: float = 0.30

    # spatial generation
    zip_band_probs: tuple[float, float, float] = (0.45, 0.40, 0.15)
    near_range_miles: tuple[float, float] = (5.0, 65.0)
    mid_range_miles: tuple[float, float] = (65.0, 130.0)
    stretch_range_miles: tuple[float, float] = (130.0, 175.0)

    # parts
    price_log_mean: float = 140.0
    price_sigma: float = 0.9
    price_min: float = 8.0
    price_max: float = 5000.0
    popularity_alpha: float = 2.0
    popularity_beta: float = 18.0

    # zip customer load
    active_customers_log_mean: float = 35.0
    active_customers_sigma: float = 0.9
    active_customers_min: int = 3
    active_customers_max: int = 1200

    # demand probability + dispatch
    demand_p_floor: float = 0.01
    demand_p_scale: float = 1.5
    demand_p_ceiling: float = 0.35
    dispatch_base: float = 0.25
    dispatch_pop_weight: float = 7.5
    dispatch_cust_sqrt_weight: float = 0.02

    # hub size / B_j assignment
    small_threshold: float = 0.35
    medium_threshold: float = 0.80
    b_share_small: float = 0.35
    b_share_medium: float = 0.60
    b_share_large: float = 0.85

    # Anchor-city scenario override. Blank/None = use DEFAULT_ANCHOR_CITY_FILE.
    anchor_city_file: str | None = None


@dataclass
class ModelParams:
    C: float = 50.0
    h_s: float = 0.6
    h_d: float = 0.6
    d_s: float = 110.0
    L: float = 50000.0
    S_lim: float = 500000.0
    S_var: float = 12.0
    lambda_1: float = 1.0
    lambda_2: float = 1.0
    lambda_3: float = 0.25
    base_miles: float = 20.0
    penalty_start_miles: float = 130.0
    max_service_miles: float = 180.0

## Section 4: Loading a Batch CSV Row into Configs

`batch_runs.csv` has one row per instance. Columns are optional; anything
missing falls back to the dataclass default, so existing batch files without
`anchor_city_file` continue to work unchanged (they'll use the default
anchor CSV).

Two helpers:

- **`_pick`** — typed fetch for scalar defaults (int / float / str).
- **`_pick_optional_str`** — used for fields whose default is `None`
  (type-agnostic), e.g. `anchor_city_file`. Returns a trimmed string or
  `None` if the cell is missing / blank.


In [4]:
def _pick(row: pd.Series, key: str, default):
    if key in row.index and pd.notna(row[key]):
        val = row[key]
        return type(default)(val) if not isinstance(default, tuple) else val
    return default


def _pick_optional_str(row: pd.Series, key: str) -> str | None:
    if key not in row.index:
        return None
    val = row[key]
    if pd.isna(val):
        return None
    s = str(val).strip()
    return s if s else None


def configs_from_row(row: pd.Series) -> Tuple[GeneratorConfig, ModelParams]:
    d_gc = GeneratorConfig(instance_name="_tmp")
    d_mp = ModelParams()

    gc = GeneratorConfig(
        instance_name=str(row["instance_name"]),
        seed=int(_pick(row, "seed", d_gc.seed)),
        schema_version=str(_pick(row, "schema_version", d_gc.schema_version)),
        n_hubs=int(_pick(row, "n_hubs", d_gc.n_hubs)),
        n_zips=int(_pick(row, "n_zips", d_gc.n_zips)),
        n_parts=int(_pick(row, "n_parts", d_gc.n_parts)),
        candidate_fraction=float(_pick(row, "candidate_fraction", d_gc.candidate_fraction)),
        zip_band_probs=(
            float(_pick(row, "zip_band_near_prob", d_gc.zip_band_probs[0])),
            float(_pick(row, "zip_band_mid_prob", d_gc.zip_band_probs[1])),
            float(_pick(row, "zip_band_stretch_prob", d_gc.zip_band_probs[2])),
        ),
        near_range_miles=(
            float(_pick(row, "near_min_miles", d_gc.near_range_miles[0])),
            float(_pick(row, "near_max_miles", d_gc.near_range_miles[1])),
        ),
        mid_range_miles=(
            float(_pick(row, "mid_min_miles", d_gc.mid_range_miles[0])),
            float(_pick(row, "mid_max_miles", d_gc.mid_range_miles[1])),
        ),
        stretch_range_miles=(
            float(_pick(row, "stretch_min_miles", d_gc.stretch_range_miles[0])),
            float(_pick(row, "stretch_max_miles", d_gc.stretch_range_miles[1])),
        ),
        price_log_mean=float(_pick(row, "price_log_mean", d_gc.price_log_mean)),
        price_sigma=float(_pick(row, "price_sigma", d_gc.price_sigma)),
        price_min=float(_pick(row, "price_min", d_gc.price_min)),
        price_max=float(_pick(row, "price_max", d_gc.price_max)),
        popularity_alpha=float(_pick(row, "popularity_alpha", d_gc.popularity_alpha)),
        popularity_beta=float(_pick(row, "popularity_beta", d_gc.popularity_beta)),
        active_customers_log_mean=float(_pick(row, "active_customers_log_mean", d_gc.active_customers_log_mean)),
        active_customers_sigma=float(_pick(row, "active_customers_sigma", d_gc.active_customers_sigma)),
        active_customers_min=int(_pick(row, "active_customers_min", d_gc.active_customers_min)),
        active_customers_max=int(_pick(row, "active_customers_max", d_gc.active_customers_max)),
        demand_p_floor=float(_pick(row, "demand_p_floor", d_gc.demand_p_floor)),
        demand_p_scale=float(_pick(row, "demand_p_scale", d_gc.demand_p_scale)),
        demand_p_ceiling=float(_pick(row, "demand_p_ceiling", d_gc.demand_p_ceiling)),
        dispatch_base=float(_pick(row, "dispatch_base", d_gc.dispatch_base)),
        dispatch_pop_weight=float(_pick(row, "dispatch_pop_weight", d_gc.dispatch_pop_weight)),
        dispatch_cust_sqrt_weight=float(_pick(row, "dispatch_cust_sqrt_weight", d_gc.dispatch_cust_sqrt_weight)),
        small_threshold=float(_pick(row, "small_threshold", d_gc.small_threshold)),
        medium_threshold=float(_pick(row, "medium_threshold", d_gc.medium_threshold)),
        b_share_small=float(_pick(row, "b_share_small", d_gc.b_share_small)),
        b_share_medium=float(_pick(row, "b_share_medium", d_gc.b_share_medium)),
        b_share_large=float(_pick(row, "b_share_large", d_gc.b_share_large)),
        anchor_city_file=_pick_optional_str(row, "anchor_city_file"),
    )

    mp = ModelParams(
        C=float(_pick(row, "C", d_mp.C)),
        h_s=float(_pick(row, "h_s", d_mp.h_s)),
        h_d=float(_pick(row, "h_d", d_mp.h_d)),
        d_s=float(_pick(row, "d_s", d_mp.d_s)),
        L=float(_pick(row, "L", d_mp.L)),
        S_lim=float(_pick(row, "S_lim", d_mp.S_lim)),
        S_var=float(_pick(row, "S_var", d_mp.S_var)),
        lambda_1=float(_pick(row, "lambda_1", d_mp.lambda_1)),
        lambda_2=float(_pick(row, "lambda_2", d_mp.lambda_2)),
        lambda_3=float(_pick(row, "lambda_3", d_mp.lambda_3)),
        base_miles=float(_pick(row, "base_miles", d_mp.base_miles)),
        penalty_start_miles=float(_pick(row, "penalty_start_miles", d_mp.penalty_start_miles)),
        max_service_miles=float(_pick(row, "max_service_miles", d_mp.max_service_miles)),
    )
    return gc, mp

## Section 5: Anchor-City Loader and Validator

Three small functions that sit between the CSV file and `generate_hubs`:

1. **`resolve_anchor_city_file(gc)`** — picks `gc.anchor_city_file` when it is
   present and non-blank, otherwise falls back to `DEFAULT_ANCHOR_CITY_FILE`.
2. **`validate_anchor_cities(anchors, path, n_hubs)`** — enforces the required
   schema and value ranges. Failure modes raise `ValueError` with messages
   that include the file path so analysts can pinpoint the offending file.
3. **`load_anchor_cities(gc)`** — resolves the path, checks it exists,
   reads with pandas, validates, and returns a clean DataFrame.

Validation checks:
- required columns present,
- `anchor_id` unique,
- `lat ∈ [-90, 90]` and numeric,
- `lon ∈ [-180, 180]` and numeric,
- `region_code ∈ {NAM, LATAM, EMEA, APAC}`,
- `n_hubs ≤ number of anchor rows` (one-hub-per-city rule).


In [5]:
def resolve_anchor_city_file(gc: GeneratorConfig) -> str:
    if gc.anchor_city_file is not None:
        candidate = str(gc.anchor_city_file).strip()
        if candidate:
            return candidate
    return DEFAULT_ANCHOR_CITY_FILE


def validate_anchor_cities(anchors: pd.DataFrame, path: str, n_hubs: int) -> None:
    missing = [c for c in REQUIRED_ANCHOR_COLUMNS if c not in anchors.columns]
    if missing:
        raise ValueError(
            f"Anchor-city file '{path}' is missing required columns: {missing}. "
            f"Required columns are {list(REQUIRED_ANCHOR_COLUMNS)}."
        )

    if anchors["anchor_id"].duplicated().any():
        dups = anchors.loc[anchors["anchor_id"].duplicated(), "anchor_id"].tolist()
        raise ValueError(
            f"Anchor-city file '{path}' has duplicate anchor_id values: {dups}. "
            "anchor_id must be unique because hubs are sampled one per city."
        )

    lat = pd.to_numeric(anchors["lat"], errors="coerce")
    lon = pd.to_numeric(anchors["lon"], errors="coerce")
    if lat.isna().any() or ((lat < -90) | (lat > 90)).any():
        raise ValueError(
            f"Anchor-city file '{path}' has invalid 'lat' values; "
            "must be numeric and within [-90, 90]."
        )
    if lon.isna().any() or ((lon < -180) | (lon > 180)).any():
        raise ValueError(
            f"Anchor-city file '{path}' has invalid 'lon' values; "
            "must be numeric and within [-180, 180]."
        )

    bad_regions = sorted(set(anchors["region_code"]) - ALLOWED_REGION_CODES)
    if bad_regions:
        raise ValueError(
            f"Anchor-city file '{path}' contains unsupported region_code values "
            f"{bad_regions}. Allowed values: {sorted(ALLOWED_REGION_CODES)}."
        )

    if n_hubs > len(anchors):
        raise ValueError(
            f"n_hubs={n_hubs} exceeds the number of anchor rows "
            f"({len(anchors)}) in '{path}'. The generator samples one hub per "
            "city; add more rows to the anchor-city CSV to support larger "
            "hub counts."
        )


def load_anchor_cities(gc: GeneratorConfig) -> pd.DataFrame:
    path = resolve_anchor_city_file(gc)
    if not Path(path).is_file():
        raise ValueError(
            f"Anchor-city file '{path}' not found. Either create it, "
            "adjust GeneratorConfig.anchor_city_file, or leave the "
            "'anchor_city_file' batch column blank to use the default."
        )

    anchors = pd.read_csv(path)
    validate_anchor_cities(anchors, path, gc.n_hubs)

    cleaned = anchors.copy()
    cleaned["lat"] = pd.to_numeric(cleaned["lat"])
    cleaned["lon"] = pd.to_numeric(cleaned["lon"])
    cleaned = cleaned.reset_index(drop=True)
    return cleaned

## Section 6: Geospatial Helpers (Math Preserved Verbatim)

Three pure-math functions. **Preserved verbatim from the director-approved
V5_3 implementation** — do not modify without sign-off.


In [6]:
def haversine_miles(lat1: pd.Series, lon1: pd.Series, lat2: pd.Series, lon2: pd.Series) -> pd.Series:
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return 3958.7613 * c


def move_point_miles(lat: float, lon: float, miles: float, bearing_rad: float) -> Tuple[float, float]:
    dlat = (miles / 69.0) * math.cos(bearing_rad)
    miles_per_deg_lon = max(1e-6, 69.172 * math.cos(math.radians(lat)))
    dlon = (miles / miles_per_deg_lon) * math.sin(bearing_rad)
    return lat + dlat, lon + dlon


def infer_region(lat: float, lon: float, city: str | None = None) -> str:
    if city in LATAM_ANCHOR_CITIES:
        return "LATAM"
    if lon <= -30:
        return "NAM" if lat >= 15 else "LATAM"
    if lon <= 65:
        return "EMEA"
    return "APAC"

## Section 7a: Generating Hubs (Anchor-Driven)

Hub generation no longer samples from a hardcoded list. Instead it:

1. Calls `load_anchor_cities(gc)` to resolve + validate + load the anchor CSV.
2. Samples `gc.n_hubs` anchor rows **without replacement** via
   `rng.choice(..., replace=False)`.
3. Deterministically sorts the selection by `(city, anchor_id)` so
   `H001..HNNN` labels are stable for a given seed + anchor file.
4. Emits one hub per selected anchor, copying `lat`, `lon`, and
   `region_code` directly from the CSV.
5. Applies the existing candidate-hub logic (`T_j` ∈ {0, 1}) using
   `gc.candidate_fraction` — anchor-city customization does **not** change
   how candidates are assigned.

The hubs DataFrame carries `anchor_id` and `anchor_city` as entity-fact
metadata; both are exported in `hubs.csv` for audit.


In [7]:
def generate_hubs(rng: np.random.Generator, gc: GeneratorConfig) -> pd.DataFrame:
    anchors = load_anchor_cities(gc)

    # One hub per anchor city: sample without replacement.
    chosen_idx = rng.choice(len(anchors), size=gc.n_hubs, replace=False)
    chosen = anchors.iloc[chosen_idx].copy()

    # Deterministic sort before labeling keeps hub_id assignment stable for
    # a given seed + anchor file (matches the V5_3 alphabetic ordering).
    chosen = chosen.sort_values(["city", "anchor_id"], kind="mergesort").reset_index(drop=True)

    n_candidates = (
        max(1, int(round(gc.candidate_fraction * gc.n_hubs)))
        if gc.candidate_fraction > 0 else 0
    )
    candidate_idx = set(
        rng.choice(np.arange(gc.n_hubs), size=n_candidates, replace=False).tolist()
    ) if n_candidates else set()

    rows = []
    for idx, row in enumerate(chosen.itertuples(index=False), start=1):
        is_candidate = (idx - 1) in candidate_idx
        rows.append({
            "hub_id": f"H{idx:03d}",
            "anchor_id": row.anchor_id,
            "anchor_city": row.city,
            "lat": round(float(row.lat), 6),
            "lon": round(float(row.lon), 6),
            "region_code": row.region_code,
            "T_j": 0 if is_candidate else 1,
        })
    return pd.DataFrame(rows)

## Section 7b: Generating Parts

Unchanged. Log-normal prices, beta-distributed popularity, both fully
parameterized by `GeneratorConfig`.

In [8]:
def generate_parts(rng: np.random.Generator, gc: GeneratorConfig) -> pd.DataFrame:
    part_ids = [f"P{k:04d}" for k in range(1, gc.n_parts + 1)]
    prices = np.exp(rng.normal(np.log(gc.price_log_mean), gc.price_sigma, gc.n_parts))
    prices = np.clip(prices, gc.price_min, gc.price_max)
    popularity = np.round(rng.beta(gc.popularity_alpha, gc.popularity_beta, gc.n_parts), 4)
    return pd.DataFrame({
        "part_id": part_ids,
        "P_k": np.round(prices, 2),
        "_popularity_k": popularity,
    })

## Section 7c: Generating ZIPs

ZIP generation is **structurally unchanged** by the anchor-city work. ZIPs
are still scattered around generated hubs using the near/mid/stretch bands,
and each ZIP inherits `region_code` from its seed hub. The only indirect
effect: swapping `anchor_city_file` changes which hub locations exist,
which in turn shifts where ZIPs are scattered.


In [9]:
def generate_zips(rng: np.random.Generator, hubs: pd.DataFrame, gc: GeneratorConfig) -> pd.DataFrame:
    rows = []
    home_hub_idx = rng.integers(0, len(hubs), size=gc.n_zips)
    band_labels = ["near", "mid", "stretch"]
    band_probs = np.array(gc.zip_band_probs, dtype=float)
    band_probs = band_probs / band_probs.sum()

    for i in range(gc.n_zips):
        hub = hubs.iloc[int(home_hub_idx[i])]
        band = rng.choice(band_labels, p=band_probs)
        if band == "near":
            radius = float(rng.uniform(*gc.near_range_miles))
        elif band == "mid":
            radius = float(rng.uniform(*gc.mid_range_miles))
        else:
            radius = float(rng.uniform(*gc.stretch_range_miles))

        bearing = float(rng.uniform(0, 2 * math.pi))
        lat, lon = move_point_miles(float(hub["lat"]), float(hub["lon"]), radius, bearing)
        active_customers = int(np.clip(
            np.exp(rng.normal(np.log(gc.active_customers_log_mean), gc.active_customers_sigma)),
            gc.active_customers_min, gc.active_customers_max,
        ))

        rows.append({
            "zip_id": f"Z{i+1:05d}",
            "lat": round(lat, 6),
            "lon": round(lon, 6),
            "region_code": hub["region_code"],
            "_active_customers": active_customers,
            "_seed_hub_id": hub["hub_id"],
        })
    return pd.DataFrame(rows)

## Section 7d: Hub Size and `B_j` Assignment

Unchanged. Hubs are ranked by seeded customer load, bucketed into
small/medium/large, and `B_j` is set as a share of `n_parts`.

In [10]:
def assign_hub_size_and_capacity(hubs: pd.DataFrame, zips: pd.DataFrame, gc: GeneratorConfig) -> pd.DataFrame:
    if not 0 < gc.small_threshold < gc.medium_threshold < 1:
        raise ValueError("Require 0 < small_threshold < medium_threshold < 1.")

    b_share_by_size = {
        "small": gc.b_share_small,
        "medium": gc.b_share_medium,
        "large": gc.b_share_large,
    }

    seeded_load = (
        zips.groupby("_seed_hub_id")["_active_customers"]
        .sum()
        .reindex(hubs["hub_id"], fill_value=0)
        .astype(int)
    )
    load_rank = seeded_load.rank(method="first", pct=True)

    out = hubs.copy()
    out["hub_size"] = np.select(
        [load_rank <= gc.small_threshold, load_rank <= gc.medium_threshold],
        ["small", "medium"],
        default="large",
    )
    out["B_j"] = out["hub_size"].map(
        lambda s: max(1, min(gc.n_parts, int(math.ceil(gc.n_parts * b_share_by_size[s]))))
    ).astype(int)
    return out

## Section 8: Distances and Feasibility Check

Unchanged. Director-approved cross-join + vectorized haversine. `d_ij` is
**raw** distance per the base-package spec (no `- 20` subtraction baked in).


In [11]:
def generate_distances(zips: pd.DataFrame, hubs: pd.DataFrame) -> pd.DataFrame:
    z = zips[["zip_id", "lat", "lon"]].copy()
    h = hubs[["hub_id", "lat", "lon"]].copy()
    cross = z.assign(_tmp=1).merge(h.assign(_tmp=1), on="_tmp", suffixes=("_zip", "_hub")).drop(columns=["_tmp"])
    cross["d_ij"] = np.round(
        haversine_miles(cross["lat_zip"], cross["lon_zip"], cross["lat_hub"], cross["lon_hub"]),
        2,
    )
    return cross[["zip_id", "hub_id", "d_ij"]]


def verify_feasibility(distances: pd.DataFrame, mp: ModelParams) -> dict:
    per_zip_min = distances.groupby("zip_id")["d_ij"].min()
    stranded = int((per_zip_min > mp.max_service_miles).sum())
    if stranded > 0:
        raise RuntimeError(
            f"{stranded} ZIPs lack any hub within {mp.max_service_miles} miles."
        )
    within_penalty = float((per_zip_min <= mp.penalty_start_miles).mean())
    within_max = float((per_zip_min <= mp.max_service_miles).mean())
    avg_eligible = float(
        distances.assign(elig=(distances["d_ij"] <= mp.max_service_miles).astype(int))
                 .groupby("zip_id")["elig"].sum().mean()
    )
    return {
        "within_penalty_start_share": within_penalty,
        "within_max_service_share": within_max,
        "avg_eligible_hubs_per_zip": avg_eligible,
    }

## Section 9: Demand Simulation, Baseline Part Homes, Parameter Key

All unchanged.

In [12]:
def generate_demand(rng: np.random.Generator, zips: pd.DataFrame, parts: pd.DataFrame, gc: GeneratorConfig) -> pd.DataFrame:
    base = (
        zips[["zip_id", "_active_customers"]].assign(_tmp=1)
        .merge(parts[["part_id", "_popularity_k"]].assign(_tmp=1), on="_tmp")
        .drop(columns=["_tmp"])
    )
    max_cust = int(zips["_active_customers"].max())
    cust_factor = np.log1p(base["_active_customers"]) / np.log(1 + max_cust)

    p = gc.demand_p_floor + gc.demand_p_scale * base["_popularity_k"] * cust_factor
    p = np.clip(p, gc.demand_p_floor, gc.demand_p_ceiling)

    q = rng.binomial(1, p.to_numpy())
    mean_dispatch = (
        gc.dispatch_base
        + gc.dispatch_pop_weight * base["_popularity_k"]
        + gc.dispatch_cust_sqrt_weight * np.sqrt(base["_active_customers"])
    )
    dispatches = np.where(q == 1, 1 + rng.poisson(mean_dispatch.to_numpy()), 0)

    base["Q_ik"] = q
    base["b_ik"] = dispatches.astype(int)
    return (
        base.loc[base["Q_ik"] == 1, ["zip_id", "part_id", "Q_ik", "b_ik"]]
            .reset_index(drop=True)
    )


def generate_baseline_part_homes(rng: np.random.Generator, hubs: pd.DataFrame, parts: pd.DataFrame) -> pd.DataFrame:
    active = hubs.loc[hubs["T_j"] == 1, "hub_id"].tolist()
    if not active:
        active = hubs["hub_id"].tolist()
    homes = rng.choice(active, size=len(parts), replace=True)
    return pd.DataFrame({"part_id": parts["part_id"], "current_home_hub": homes})


def parameter_key_table() -> pd.DataFrame:
    rows = [
        ("i", "Index", "ZIP space"),
        ("j", "Index", "Hub space"),
        ("k", "Index", "Part space"),
        ("hub_id", "Metadata", "Unique hub identifier"),
        ("anchor_id", "Metadata", "Anchor-city identifier sourced from the anchor-city CSV"),
        ("anchor_city", "Metadata", "Human-readable anchor-city name sourced from the anchor-city CSV"),
        ("zip_id", "Metadata", "Unique ZIP identifier"),
        ("part_id", "Metadata", "Unique part identifier"),
        ("region_code", "Metadata", "Operating region code (NAM/LATAM/EMEA/APAC)"),
        ("T_j", "Indexed parameter", "Binary hub state (1 = active, 0 = candidate)"),
        ("B_j", "Indexed parameter", "Hub capacity / stocking bound"),
        ("P_k", "Indexed parameter", "Part-level price / stocking cost"),
        ("d_ij", "Indexed parameter", "Raw haversine distance ZIP i to hub j (miles)"),
        ("Q_ik", "Indexed parameter", "Binary demand indicator"),
        ("b_ik", "Indexed parameter", "Dispatch count / demand intensity"),
        ("C", "Global scalar", "Transfer-cost scalar"),
        ("h_s", "Global scalar", "Base transport coefficient"),
        ("h_d", "Global scalar", "Distance transport coefficient"),
        ("d_s", "Global scalar", "Distance threshold / service-penalty start (model)"),
        ("L", "Global scalar", "Soft stocking limit per hub"),
        ("S_lim", "Global scalar", "Base fixed cost for keeping a hub open"),
        ("S_var", "Global scalar", "Overflow cost per stocked part beyond L"),
        ("lambda_1", "Global scalar", "Weight on base transport term"),
        ("lambda_2", "Global scalar", "Weight on distance transport term"),
        ("lambda_3", "Global scalar", "Weight on distance-above-threshold term"),
        ("base_miles", "Global scalar", "Base / free-miles distance threshold"),
        ("penalty_start_miles", "Global scalar", "Distance at which SLA penalty starts"),
        ("max_service_miles", "Global scalar", "Maximum eligible service distance"),
    ]
    return pd.DataFrame(rows, columns=["Symbol", "Type", "Description"])

## Section 10: Pipeline Orchestrator

`run_instance` now captures the resolved anchor-city path and its row count
up front so `summary_report.csv` can record them alongside
`n_hubs_requested` and `n_hubs_generated` — the four new audit fields
requested by the design.


In [13]:
def _clear_directory(path: Path) -> None:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def run_instance(gc: GeneratorConfig, mp: ModelParams, out_root: Path) -> dict:
    rng = np.random.default_rng(gc.seed)

    # Resolve + load anchor cities up front so summary audit fields reflect
    # the exact file the generator consumed.
    anchor_city_file = resolve_anchor_city_file(gc)
    anchor_city_count = len(load_anchor_cities(gc))

    hubs_raw = generate_hubs(rng, gc)
    parts_raw = generate_parts(rng, gc)
    zips_raw = generate_zips(rng, hubs_raw, gc)
    hubs_raw = assign_hub_size_and_capacity(hubs_raw, zips_raw, gc)

    distances = generate_distances(zips_raw, hubs_raw)
    feas = verify_feasibility(distances, mp)

    demand = generate_demand(rng, zips_raw, parts_raw, gc)
    baseline = generate_baseline_part_homes(rng, hubs_raw, parts_raw)

    # anchor_id and anchor_city are entity-fact metadata, not preprocessing
    # artifacts, so they are retained in hubs.csv for auditability.
    hubs_out = hubs_raw[[
        "hub_id", "anchor_id", "anchor_city", "lat", "lon",
        "region_code", "T_j", "B_j",
    ]].copy()
    parts_out = parts_raw[["part_id", "P_k"]].copy()
    zips_out = zips_raw[["zip_id", "lat", "lon", "region_code"]].copy()
    demand_out = demand[["zip_id", "part_id", "Q_ik", "b_ik"]].copy()
    distances_out = distances[["zip_id", "hub_id", "d_ij"]].copy()
    parameters_out = pd.DataFrame([asdict(mp)])

    out_dir = out_root / gc.instance_name
    _clear_directory(out_dir)

    parameters_out.to_csv(out_dir / "parameters.csv", index=False)
    hubs_out.to_csv(out_dir / "hubs.csv", index=False)
    parts_out.to_csv(out_dir / "parts.csv", index=False)
    zips_out.to_csv(out_dir / "zips.csv", index=False)
    demand_out.to_csv(out_dir / "demand.csv", index=False)
    distances_out.to_csv(out_dir / "distances.csv", index=False)

    baseline.to_csv(out_dir / "optional_baseline_part_homes.csv", index=False)
    parameter_key_table().to_csv(out_dir / "optional_parameter_key.csv", index=False)

    summary = {
        "instance_name": gc.instance_name,
        "schema_version": gc.schema_version,
        "seed": gc.seed,
        "anchor_city_file": anchor_city_file,
        "anchor_city_count": anchor_city_count,
        "n_hubs_requested": gc.n_hubs,
        "n_hubs_generated": len(hubs_out),
        "n_hubs": len(hubs_out),
        "n_zips": len(zips_out),
        "n_parts": len(parts_out),
        "n_demand_rows": len(demand_out),
        "n_distance_rows": len(distances_out),
        **feas,
    }
    pd.DataFrame([summary]).to_csv(out_dir / "summary_report.csv", index=False)

    print(f"[{gc.instance_name}] anchors={anchor_city_count} "
          f"hubs={summary['n_hubs_generated']}/{summary['n_hubs_requested']} "
          f"zips={summary['n_zips']} parts={summary['n_parts']} "
          f"demand_rows={summary['n_demand_rows']} "
          f"within_max_service={summary['within_max_service_share']:.1%} "
          f"-> {out_dir}")
    return summary


def run_batch(batch_csv: Path, out_root: Path) -> pd.DataFrame:
    batch = pd.read_csv(batch_csv)
    if "instance_name" not in batch.columns:
        raise ValueError("batch CSV must contain an 'instance_name' column")

    out_root.mkdir(parents=True, exist_ok=True)
    summaries = []
    for _, row in batch.iterrows():
        gc, mp = configs_from_row(row)
        summaries.append(run_instance(gc, mp, out_root))

    batch_summary = pd.DataFrame(summaries)
    batch_summary.to_csv(out_root / "batch_summary.csv", index=False)
    return batch_summary

## Section 11: Execution

Point `BATCH_CSV` at your configuration file and run. The sample
`batch_runs.csv` ships with four rows — three use the default anchor file
(blank `anchor_city_file` column) and one exercises the LATAM-only scenario
via `configs/anchor_cities/latam_anchor_cities.csv`.


In [16]:
BATCH_CSV = Path("batch_runs.csv")
OUT_ROOT = Path("outputs")

batch_summary = run_batch(BATCH_CSV, OUT_ROOT)
batch_summary

[sandbox_small] anchors=100 hubs=10/10 zips=50 parts=25 demand_rows=126 within_max_service=100.0% -> outputs/sandbox_small
[midscale_balanced] anchors=100 hubs=40/40 zips=1000 parts=150 demand_rows=13859 within_max_service=100.0% -> outputs/midscale_balanced
[latam_scenario] anchors=10 hubs=8/8 zips=400 parts=100 demand_rows=4129 within_max_service=100.0% -> outputs/latam_scenario
[scale_stretchy_sla] anchors=100 hubs=100/100 zips=2000 parts=250 demand_rows=80970 within_max_service=100.0% -> outputs/scale_stretchy_sla


,instance_name,schema_version,seed,anchor_city_file,anchor_city_count,n_hubs_requested,n_hubs_generated,n_hubs,n_zips,n_parts,n_demand_rows,n_distance_rows,within_penalty_start_share,within_max_service_share,avg_eligible_hubs_per_zip
0,sandbox_small,1.0.0,1,/content/default_anchor_cities.csv,100,10,10,10,50,25,126,500,0.9200,1.0,1.0000
1,midscale_balanced,1.0.0,7,/content/default_anchor_cities.csv,100,40,40,40,1000,150,13859,40000,0.8750,1.0,1.1790
2,latam_scenario,1.0.0,11,/content/latam_anchor_cities.csv,10,8,8,8,400,100,4129,3200,0.8825,1.0,1.0375
3,scale_stretchy_sla,1.0.0,42,/content/default_anchor_cities.csv,100,100,100,100,2000,250,80970,200000,0.8175,1.0,1.5500


## Output Data Dictionary

### Base package (required)
| File | Columns |
|---|---|
| `parameters.csv` | `C, h_s, h_d, d_s, L, S_lim, S_var, lambda_1-3, base_miles, penalty_start_miles, max_service_miles` |
| `hubs.csv` | `hub_id, anchor_id, anchor_city, lat, lon, region_code, T_j, B_j` |
| `parts.csv` | `part_id, P_k` |
| `zips.csv` | `zip_id, lat, lon, region_code` |
| `demand.csv` | `zip_id, part_id, Q_ik, b_ik` |
| `distances.csv` | `zip_id, hub_id, d_ij` (raw miles) |

### Auxiliary outputs (optional, clearly prefixed)
| File | Purpose |
|---|---|
| `optional_baseline_part_homes.csv` | Status-quo part assignments for baseline cost comparisons. |
| `optional_parameter_key.csv` | Self-documenting symbol reference table. |
| `summary_report.csv` | Per-instance QA: row counts, feasibility stats, and anchor-city audit fields (`anchor_city_file`, `anchor_city_count`, `n_hubs_requested`, `n_hubs_generated`). |

### Dropped during export (internal-only, kept in code)
- `_popularity_k`, `_active_customers`, `_seed_hub_id`, `hub_size` — generator
  scratch fields used to shape demand and assign `B_j`.
- Eligibility flags or thresholded distance columns. Those belong in the
  modeling team's preprocessing layer, not in the base package.
